# 05 — Master Pipeline (Twitter-RoBERTa): CEO Self-Presentation Discrepancy

**Cross-validation companion** to the FinBERT master notebook (`04_master_pipeline_finbert_colab.ipynb`). Produces transcript-side **Twitter-RoBERTa** scores (`cardiffnlp/twitter-roberta-base-sentiment-latest`) so CEO rankings can be compared across models.

**Cross-model discrepancy note:** the Reddit-side aggregate you have (`ceo_year_finbert_abc_merged_updated.csv`) is FinBERT-scored only. The discrepancy this notebook computes is therefore `roberta_self − finbert_crowd` — a **mixed-model** measure useful as a robustness check, not a strict same-model comparison. To get a same-model `roberta_self − roberta_crowd`, re-score the Reddit comments with RoBERTa first and produce an analogous merged CSV.

**Pipeline stages:**

| Stage | Purpose | Runtime (T4) |
|---|---|---|
| A. Load raw data | transcripts + CEO universe + dictionaries + Reddit aggregate | 1 min |
| B. Extract CEO utterances | (skipped if cache exists) | 2 min |
| C. Dictionary scoring | (skipped if cache exists) | 2 min |
| D. RoBERTa scoring | sliding-window chunking for long utterances | 40-60 min |
| E. Aggregate | per (execid, year, quarter, section) + (execid, year, section) | 1 min |
| F. Discrepancy join | mixed-model: roberta_self − finbert_crowd | 1 min |
| G. Save & download | parquet + CSV | 1 min |

**Cache-aware:** if `ceo_utterances_dict_scored.parquet` already exists on Drive (e.g., from running the FinBERT notebook first), Stages B and C are skipped — only RoBERTa scoring runs.

**Required uploads to `/content/drive/MyDrive/ceo_reddit/data/`** (combined ~10 MB):
- `ceo_universe.parquet`
- `Loughran-McDonald_MasterDictionary_1993-2025.csv`
- `CEO_Integrity_Dictionary.csv`
- `CEO_Narcissism_Dictionary.csv`
- `ceo_year_finbert_abc_merged_updated.csv`

**Auto-fetched at runtime (no upload needed):**
- `earnings_transcripts.parquet` (~2.5 GB) — pulled from HuggingFace `Bose345/sp500_earnings_transcripts` via Colab's gigabit pipe. Cached to Drive automatically if `TRANSCRIPTS_DRIVE` is later uploaded.

## Setup

In [ ]:
import torch
if torch.cuda.is_available():
    device = torch.device("cuda")
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    device = torch.device("cpu")
    print("WARNING: No GPU — Runtime > Change runtime type > T4 GPU")

In [ ]:
!pip install -q transformers fastparquet datasets

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import shutil
DRIVE_DATA = Path("/content/drive/MyDrive/ceo_reddit/data")
LOCAL_DIR = Path("/content/ceo_reddit")
LOCAL_DIR.mkdir(exist_ok=True)

# Files you upload to Drive (combined ~10 MB — easy via web UI)
REQUIRED_DRIVE = [
    "ceo_universe.parquet",
    "Loughran-McDonald_MasterDictionary_1993-2025.csv",
    "CEO_Integrity_Dictionary.csv",
    "CEO_Narcissism_Dictionary.csv",
    "ceo_year_finbert_abc_merged_updated.csv",
]
missing = [f for f in REQUIRED_DRIVE if not (DRIVE_DATA / f).exists()]
assert not missing, f"Missing on Drive: {missing}"
for f in REQUIRED_DRIVE:
    src, dst = DRIVE_DATA / f, LOCAL_DIR / f
    if not dst.exists():
        shutil.copy2(src, dst)
        print(f"Copied {f}: {src.stat().st_size / 1024 / 1024:.1f} MB")

# earnings_transcripts.parquet: prefer Drive cache; otherwise fetch from HuggingFace.
# The HF download is ~2.5 GB and runs on Colab's gigabit pipe (5-10 min) instead of
# uploading from your machine. Saved to local Colab disk only — not written back to
# Drive (Colab→Drive write of 2.5 GB is throttled by API limits and takes longer than
# the HF re-fetch on a future session).
TRANSCRIPTS_LOCAL = LOCAL_DIR / "earnings_transcripts.parquet"
TRANSCRIPTS_DRIVE = DRIVE_DATA / "earnings_transcripts.parquet"
if TRANSCRIPTS_LOCAL.exists():
    print(f"Transcripts already on local disk: {TRANSCRIPTS_LOCAL.stat().st_size / 1024 / 1024:.1f} MB")
elif TRANSCRIPTS_DRIVE.exists():
    print("Found transcripts on Drive — copying to local disk...")
    shutil.copy2(TRANSCRIPTS_DRIVE, TRANSCRIPTS_LOCAL)
    print(f"  copied {TRANSCRIPTS_LOCAL.stat().st_size / 1024 / 1024:.1f} MB")
else:
    print("Fetching earnings transcripts from HuggingFace (Bose345/sp500_earnings_transcripts)...")
    from datasets import load_dataset
    ds = load_dataset("Bose345/sp500_earnings_transcripts", split="train")
    ds.to_parquet(str(TRANSCRIPTS_LOCAL))
    print(f"  saved {TRANSCRIPTS_LOCAL.stat().st_size / 1024 / 1024:.1f} MB to local disk")


## Cache check

If a previous run (this notebook or the FinBERT one) already produced `ceo_utterances_dict_scored.parquet`, we load it and skip Stages B+C — saves ~5 min when iterating.

In [ ]:
import pandas as pd
import numpy as np

DICT_SCORED_PATH = DRIVE_DATA / "ceo_utterances_dict_scored.parquet"
if DICT_SCORED_PATH.exists():
    df = pd.read_parquet(DICT_SCORED_PATH)
    SKIP_EXTRACTION = True
    print(f"Loaded cached dict-scored utterances: {len(df):,} rows — skipping Stages B+C")
    print(f"Sections: {df.section.value_counts().to_dict()}")
else:
    df = None
    SKIP_EXTRACTION = False
    print("No cache — will run Stages B + C from scratch")

## Stages B + C — Extraction & Dictionary Scoring

Skipped automatically if cache loaded above. The cell below performs:
1. Extract CEO-only utterances from `structured_content` using last-name match + exec/analyst guard + structural Q&A boundary detection.
2. Score each utterance with Loughran-McDonald + Hennig integrity + narcissism dictionaries.
3. Save `ceo_utterances_dict_scored.parquet` to Drive as a checkpoint.

In [ ]:
import json, re, time
from datetime import date, datetime

if not SKIP_EXTRACTION:
    DASH_SPLIT_RE = re.compile(r"\s[-–—]\s+")
    EXEC_TITLE_KEYWORDS = (
        "officer", "president", "ceo", "cfo", "coo", "cio", "cto", "cmo",
        "chairman", "chairwoman", "chairperson", "chair",
        "founder", "co-founder", "director", "managing director",
        "vp", "evp", "svp", "vice president", "head of",
        "treasurer", "controller", "secretary",
        "investor relations", "ir contact", "general counsel",
        "executive", "principal", "owner",
    )
    QA_CUE_RE = re.compile(
        r"first\s+question\s+(?:comes|is|will\s+come|today\s+is)\s+from"
        r"|now\s+(?:begin|take|start|open)\s+(?:the\s+|our\s+)?(?:question[-\s]and[-\s]answer|q\s*&\s*a)"
        r"|now\s+take\s+(?:our|the|your)?\s*first\s+question"
        r"|will\s+now\s+take\s+(?:any\s+|your\s+|some\s+)?questions"
        r"|open\s+(?:up\s+)?(?:the\s+)?(?:floor|line|call)\s+(?:up\s+)?(?:to|for)\s+(?:any\s+|your\s+)?questions"
        r"|opening\s+(?:up\s+)?(?:the\s+)?(?:floor|line)\s+(?:up\s+)?(?:to|for)\s+questions"
        r"|turn\s+(?:it\s+)?(?:back\s+)?(?:over\s+)?to\s+the\s+operator\s+for\s+(?:questions|q\s*&\s*a)"
        r"|operator\s*,?\s+(?:please\s+)?(?:provide\s+(?:the\s+)?instructions\s+for|let'?s\s+take|take\s+(?:the\s+)?(?:first\s+)?question)"
        r"|please\s+(?:provide|give\s+us)\s+(?:the\s+)?instructions\s+for\s+(?:the\s+)?q\s*&\s*a",
        re.IGNORECASE,
    )

    def looks_like_analyst(speaker):
        if not speaker or not DASH_SPLIT_RE.search(speaker):
            return False
        after = DASH_SPLIT_RE.split(speaker, maxsplit=1)[-1].strip().lower()
        return bool(after) and not any(kw in after for kw in EXEC_TITLE_KEYWORDS)

    def normalize_structured(value):
        if value is None: return []
        if isinstance(value, str):
            try: parsed = json.loads(value)
            except json.JSONDecodeError: return []
            return parsed if isinstance(parsed, list) else []
        try: return [item for item in value if isinstance(item, dict)]
        except TypeError: return []

    def detect_qa_start(utterances):
        saw_first_op, saw_exec = False, False
        for idx, item in enumerate(utterances):
            sp = (item.get("speaker") or "").strip()
            tx = (item.get("text") or "").strip()
            is_op = sp.lower() == "operator"
            if is_op and len(tx) >= 30:
                if saw_first_op and saw_exec: return idx
                saw_first_op = True
            elif sp and not is_op and len(tx) >= 30:
                saw_exec = True
        for idx, item in enumerate(utterances):
            if QA_CUE_RE.search(item.get("text") or ""): return idx + 1
        for idx, item in enumerate(utterances):
            if looks_like_analyst((item.get("speaker") or "").strip()): return idx
        return len(utterances)

    def coerce_date(raw):
        if raw is None: return None
        if isinstance(raw, date) and not isinstance(raw, datetime): return raw
        if isinstance(raw, datetime): return raw.date()
        if isinstance(raw, str):
            try: return datetime.fromisoformat(raw.replace("Z", "+00:00")).date()
            except ValueError: return None
        return None

    transcripts = pd.read_parquet(LOCAL_DIR / "earnings_transcripts.parquet")
    ceo_universe = pd.read_parquet(LOCAL_DIR / "ceo_universe.parquet")
    for col in ("ceo_start_date", "ceo_end_date"):
        ceo_universe[col] = pd.to_datetime(ceo_universe[col], errors="coerce")
    transcripts["call_date"] = transcripts["date"].map(coerce_date)
    print(f"Loaded {len(transcripts):,} transcripts and {len(ceo_universe):,} CEO records")

    all_rows = []
    diag = {"n_with_tenured_ceo": 0, "n_matched": 0, "n_no_qa": 0}
    t0 = time.time()
    for _, row in transcripts.iterrows():
        sc = normalize_structured(row["structured_content"])
        if not sc or row["call_date"] is None: continue
        mask = (
            (ceo_universe["ticker"] == row["symbol"])
            & (ceo_universe["ceo_start_date"].dt.date <= row["call_date"])
            & (ceo_universe["ceo_end_date"].isna() | (ceo_universe["ceo_end_date"].dt.date >= row["call_date"]))
        )
        ceos = ceo_universe.loc[mask]
        if ceos.empty: continue
        diag["n_with_tenured_ceo"] += 1
        qa_start = detect_qa_start(sc)
        if qa_start >= len(sc): diag["n_no_qa"] += 1
        ceo_by_lname = {c["last_name"].lower(): c for _, c in ceos.iterrows() if isinstance(c["last_name"], str)}
        matched = False
        for idx, item in enumerate(sc):
            speaker = (item.get("speaker") or "").strip()
            text = (item.get("text") or "").strip()
            if not speaker or not text or looks_like_analyst(speaker): continue
            sp_lower = speaker.lower()
            ceo_match = next((c for ln, c in ceo_by_lname.items() if ln in sp_lower), None)
            if ceo_match is None: continue
            matched = True
            all_rows.append({
                "symbol": row["symbol"], "company_name": row.get("company_name"),
                "year": int(row["year"]), "quarter": int(row["quarter"]),
                "call_date": row["call_date"],
                "execid": int(ceo_match["execid"]), "ceo_full_name": ceo_match["full_name"],
                "section": "prepared" if idx < qa_start else "qa",
                "utterance_idx": idx, "speaker_label": speaker,
                "text": text, "n_words": len(text.split()),
            })
        if matched: diag["n_matched"] += 1

    df = pd.DataFrame(all_rows)
    valid = df["text"].str.len() >= 10
    df = df[valid].reset_index(drop=True)
    print(f"Stage B done in {time.time()-t0:.0f}s: {len(df):,} CEO utterances across {diag['n_matched']:,} calls")
    print(f"  match rate among tenured: {diag['n_matched']/diag['n_with_tenured_ceo']:.1%}")
    print(f"  sections: {df.section.value_counts().to_dict()}")

    # Stage C: dictionary scoring
    lm = pd.read_csv(LOCAL_DIR / "Loughran-McDonald_MasterDictionary_1993-2025.csv")
    lm_words = {
        "lm_positive": set(lm[lm["Positive"] > 0]["Word"].str.lower()),
        "lm_negative": set(lm[lm["Negative"] > 0]["Word"].str.lower()),
        "lm_uncertainty": set(lm[lm["Uncertainty"] > 0]["Word"].str.lower()),
        "lm_strong_modal": set(lm[lm["Strong_Modal"] > 0]["Word"].str.lower()),
        "lm_weak_modal": set(lm[lm["Weak_Modal"] > 0]["Word"].str.lower()),
    }
    integ = pd.read_csv(LOCAL_DIR / "CEO_Integrity_Dictionary.csv")
    integ_words = {f"integrity_{cat.lower()}": set(integ[integ["Integrity_Category"] == cat]["Word"].str.lower()) for cat in integ["Integrity_Category"].unique()}
    integ_words["integrity_all"] = set(integ["Word"].str.lower())
    narc_words = set(pd.read_csv(LOCAL_DIR / "CEO_Narcissism_Dictionary.csv")["Word"].str.lower())
    TOK = re.compile(r"[a-z]+")

    def score_text(text):
        tokens = TOK.findall(str(text).lower())
        n = len(tokens)
        if n == 0:
            return {"word_count": 0, **{k: 0 for k in (*lm_words, *integ_words, "narcissism")},
                    "lm_net_sentiment": 0.0, "lm_overconfidence": 0.0,
                    "integrity_norm": 0.0, "narcissism_norm": 0.0}
        counts = {k: sum(1 for t in tokens if t in s) for k, s in lm_words.items()}
        counts.update({k: sum(1 for t in tokens if t in s) for k, s in integ_words.items()})
        counts["narcissism"] = sum(1 for t in tokens if t in narc_words)
        pos, neg, strong, unc = counts["lm_positive"], counts["lm_negative"], counts["lm_strong_modal"], counts["lm_uncertainty"]
        return {
            "word_count": n, **counts,
            "lm_net_sentiment": round((pos - neg) / n, 6),
            "lm_overconfidence": round((pos + strong - unc) / n, 6),
            "integrity_norm": round(counts["integrity_all"] / n, 6),
            "narcissism_norm": round(counts["narcissism"] / n, 6),
        }

    t0 = time.time()
    scores = pd.DataFrame([score_text(t) for t in df["text"].values])
    df = pd.concat([df, scores], axis=1)
    print(f"Stage C done in {time.time()-t0:.0f}s")
    print(df.groupby("section")[["lm_overconfidence", "lm_net_sentiment", "integrity_norm"]].mean().round(4))

    df.to_parquet(DICT_SCORED_PATH, engine="fastparquet", compression="zstd", index=False)
    print(f"\nCheckpoint saved: {DICT_SCORED_PATH}")

## Stage D — Twitter-RoBERTa Scoring (sliding-window chunking)

Same chunking strategy as the FinBERT notebook (510-token windows, 50-token overlap, weighted-averaged back to utterance level). RoBERTa's BPE tokenizer counts tokens differently from FinBERT's BERT tokenizer, so we re-tokenize from scratch.

**Important: RoBERTa label order differs from FinBERT.** RoBERTa: `0=negative, 1=neutral, 2=positive`. FinBERT: `0=positive, 1=negative, 2=neutral`. We re-derive label/sentiment columns accordingly.

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch.nn.functional as F

WINDOW_TOKENS, OVERLAP_TOKENS, BATCH_SIZE = 510, 50, 32

def build_chunks(texts, tokenizer):
    chunk_utt, chunk_texts, chunk_lens = [], [], []
    step = WINDOW_TOKENS - OVERLAP_TOKENS
    for utt_idx, text in enumerate(texts):
        ids = tokenizer.encode(str(text), add_special_tokens=False, truncation=False)
        if len(ids) <= WINDOW_TOKENS:
            chunk_utt.append(utt_idx); chunk_texts.append(str(text)); chunk_lens.append(max(len(ids), 1))
            continue
        for start in range(0, len(ids), step):
            window = ids[start : start + WINDOW_TOKENS]
            if not window: break
            chunk_utt.append(utt_idx); chunk_texts.append(tokenizer.decode(window, skip_special_tokens=True)); chunk_lens.append(len(window))
            if start + WINDOW_TOKENS >= len(ids): break
    return np.array(chunk_utt), chunk_texts, np.array(chunk_lens, dtype=np.float32)

def aggregate_chunks(chunk_utt, chunk_lens, chunk_probs, n_utt):
    out = np.zeros((n_utt, chunk_probs.shape[1]), dtype=np.float32)
    weights = np.zeros(n_utt, dtype=np.float32)
    np.add.at(out, chunk_utt, chunk_probs * chunk_lens[:, None])
    np.add.at(weights, chunk_utt, chunk_lens)
    weights = np.where(weights == 0, 1.0, weights)
    return out / weights[:, None]

In [ ]:
MODEL_NAME = "cardiffnlp/twitter-roberta-base-sentiment-latest"
tk = AutoTokenizer.from_pretrained(MODEL_NAME)
md = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME).to(device).eval()
print(f"Loaded {MODEL_NAME}, id2label={md.config.id2label}")

t0 = time.time()
chunk_utt, chunk_texts, chunk_lens = build_chunks(df["text"].values, tk)
print(f"  {len(df):,} utterances → {len(chunk_texts):,} chunks (avg {len(chunk_texts)/len(df):.2f}); tokenize {time.time()-t0:.0f}s")

In [ ]:
CKPT_DIR = DRIVE_DATA / "roberta_utt_checkpoints"
CKPT_DIR.mkdir(exist_ok=True)
ckpt = CKPT_DIR / "chunk_scores_partial.npy"
n_chunks = len(chunk_texts)
probs = np.zeros((n_chunks, 3), dtype=np.float32)
start_chunk = 0
if ckpt.exists():
    saved = np.load(ckpt)
    if saved.shape[0] <= n_chunks and saved.shape[1] == 3:
        probs[: saved.shape[0]] = saved
        start_chunk = saved.shape[0]
        print(f"Resuming from chunk {start_chunk:,} / {n_chunks:,}")

t0 = time.time()
for batch_start in range(start_chunk, n_chunks, BATCH_SIZE):
    batch_end = min(batch_start + BATCH_SIZE, n_chunks)
    with torch.no_grad():
        inp = tk(chunk_texts[batch_start:batch_end], padding=True, truncation=True, max_length=512, return_tensors="pt").to(device)
        probs[batch_start:batch_end] = F.softmax(md(**inp).logits, dim=-1).cpu().numpy()
    if batch_end % 10_000 < BATCH_SIZE:
        elapsed = time.time() - t0
        done = batch_end - start_chunk
        rate = done / elapsed if elapsed else 0
        eta = (n_chunks - batch_end) / rate / 60 if rate else 0
        print(f"    {batch_end:>7,} / {n_chunks:,} ({batch_end/n_chunks:.1%}) | {rate:.0f} chunks/sec | ETA {eta:.0f}m")
    if batch_end % 50_000 < BATCH_SIZE and batch_end > start_chunk:
        np.save(ckpt, probs[:batch_end])
np.save(ckpt, probs)
print(f"RoBERTa done: {n_chunks:,} chunks in {(time.time()-t0)/60:.1f} min")

# RoBERTa label order: 0=negative, 1=neutral, 2=positive
utt_probs = aggregate_chunks(chunk_utt, chunk_lens, probs, len(df))
df["roberta_negative"] = utt_probs[:, 0]
df["roberta_neutral"]  = utt_probs[:, 1]
df["roberta_positive"] = utt_probs[:, 2]
df["roberta_label"] = df[["roberta_negative", "roberta_neutral", "roberta_positive"]].idxmax(axis=1).str.replace("roberta_", "")
df["roberta_sentiment"] = df["roberta_positive"] - df["roberta_negative"]
print(df.groupby("section")[["roberta_sentiment", "roberta_positive", "roberta_negative"]].mean().round(4))

del md, tk
torch.cuda.empty_cache()

In [ ]:
scored_path = DRIVE_DATA / "ceo_utterances_roberta_scored.parquet"
df.to_parquet(scored_path, engine="fastparquet", compression="zstd", index=False)
print(f"Saved: {scored_path} ({scored_path.stat().st_size / 1024 / 1024:.1f} MB)")

## Stage E — Aggregate per (CEO, year, quarter, section) and (CEO, year, section)

In [ ]:
GROUP_Q = ["execid", "ceo_full_name", "symbol", "company_name", "year", "quarter", "section"]
GROUP_Y = ["execid", "ceo_full_name", "symbol", "company_name", "year", "section"]

AGG_SPEC = dict(
    n_utterances=("text", "count"),
    total_words=("word_count", "sum"),
    lm_overconfidence_self=("lm_overconfidence", "mean"),
    lm_net_sentiment_self=("lm_net_sentiment", "mean"),
    integrity_norm_self=("integrity_norm", "mean"),
    narcissism_norm_self=("narcissism_norm", "mean"),
    roberta_sentiment_self=("roberta_sentiment", "mean"),
    roberta_positive_self=("roberta_positive", "mean"),
    roberta_negative_self=("roberta_negative", "mean"),
    roberta_neutral_self=("roberta_neutral", "mean"),
    pct_roberta_positive_self=("roberta_label", lambda s: (s == "positive").mean()),
    pct_roberta_negative_self=("roberta_label", lambda s: (s == "negative").mean()),
)

agg_q = df.groupby(GROUP_Q, sort=False).agg(**AGG_SPEC).reset_index()
agg_y = df.groupby(GROUP_Y, sort=False).agg(**AGG_SPEC).reset_index()
print(f"quarter-section: {len(agg_q):,} rows | year-section: {len(agg_y):,} rows")

## Stage F — Discrepancy Join (cross-model: roberta_self − finbert_crowd)

The Reddit-side aggregate carries FinBERT scores only. We compute `roberta_self − finbert_crowd` as a robustness check; for a strict same-model comparison, re-score Reddit with RoBERTa first. The Compustat key (`gvkey`) carried over from the Reddit aggregate makes the output ready for downstream accounting joins.

In [ ]:
reddit = pd.read_csv(LOCAL_DIR / "ceo_year_finbert_abc_merged_updated.csv")
reddit_keep = reddit[[
    "execid", "year", "gvkey", "cusip", "abc_conm", "mention_count",
    "finbert_sentiment_mean", "finbert_positive_mean", "finbert_negative_mean",
    "pct_positive", "pct_negative",
]].rename(columns={
    "mention_count": "reddit_mention_count",
    "finbert_sentiment_mean": "finbert_sentiment_crowd",
    "finbert_positive_mean":  "finbert_positive_crowd",
    "finbert_negative_mean":  "finbert_negative_crowd",
    "pct_positive":           "pct_finbert_positive_crowd",
    "pct_negative":           "pct_finbert_negative_crowd",
})

final = agg_y.merge(reddit_keep, on=["execid", "year"], how="inner")
# Cross-model: roberta_self - finbert_crowd (interpret as a directional index, not a strict difference)
final["sentiment_discrepancy_xmodel"] = final["roberta_sentiment_self"] - final["finbert_sentiment_crowd"]
final["positive_discrepancy_xmodel"]  = final["roberta_positive_self"]  - final["finbert_positive_crowd"]
final["negative_discrepancy_xmodel"]  = final["roberta_negative_self"]  - final["finbert_negative_crowd"]

print(f"final: {len(final):,} rows ({final.execid.nunique()} CEOs, {final.gvkey.nunique()} gvkeys)")
print(f"sections: {final.section.value_counts().to_dict()}")
print("\n=== Cross-model discrepancy means by section ===")
print(final.groupby("section")[[
    "roberta_sentiment_self", "finbert_sentiment_crowd", "sentiment_discrepancy_xmodel"
]].mean().round(4))

In [ ]:
# Sanity: biggest cross-model discrepancies in PREPARED with non-trivial Reddit coverage
cand = final[(final.section == "prepared") & (final.reddit_mention_count >= 20)]
print("=== Largest POSITIVE x-model discrepancy (RoBERTa-self bullish, FinBERT-crowd less so) ===")
print(cand.nlargest(10, "sentiment_discrepancy_xmodel")[[
    "ceo_full_name", "symbol", "year",
    "roberta_sentiment_self", "finbert_sentiment_crowd",
    "sentiment_discrepancy_xmodel", "reddit_mention_count"
]].to_string(index=False))

print("\n=== Largest NEGATIVE x-model discrepancy ===")
print(cand.nsmallest(10, "sentiment_discrepancy_xmodel")[[
    "ceo_full_name", "symbol", "year",
    "roberta_sentiment_self", "finbert_sentiment_crowd",
    "sentiment_discrepancy_xmodel", "reddit_mention_count"
]].to_string(index=False))

## Stage G — Save & Download

In [ ]:
agg_q_path = DRIVE_DATA / "ceo_quarter_section_roberta_scores.csv"
agg_q.to_csv(agg_q_path, index=False); print(f"Saved: {agg_q_path}")

agg_y_path = DRIVE_DATA / "ceo_year_section_roberta_scores.csv"
agg_y.to_csv(agg_y_path, index=False); print(f"Saved: {agg_y_path}")

final_path = DRIVE_DATA / "ceo_year_section_roberta_discrepancy.csv"
final.to_csv(final_path, index=False); print(f"Saved: {final_path} ({len(final):,} rows)")

In [ ]:
from google.colab import files
files.download(str(final_path))